<a href="https://colab.research.google.com/github/zpsheldon/meg-neural-decoding/blob/main/calc_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [1]:
# Install additional depdendencies
%pip install -q lightning torchmetrics scikit-learn plotly ipywidgets pnpl

# Set up base path for dataset and related files (base_path is assumed to be set in the cells below!)
base_path = "./libribrain"
try:
    import google.colab  # This module is only available in Colab.
    in_colab = True
    base_path = "/content"  # This is the folder displayed in the Colab sidebar
except ImportError:
    in_colab = False

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.9/168.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.4/832.4 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 36.5 MB/s eta 0:00:00


In [2]:
from pnpl.datasets import LibriBrainSpeech
from torch.utils.data import DataLoader
import pandas as pd
import random
import numpy as np
import torch
import platform

## Load data

In [3]:
num_books = 7
num_chapters = [9, 12, 12, 12, 15, 14, 14]

In [4]:
# Conditionally set num_workers to avoid multiprocessing issues (try increasing if performance is problematic)
num_workers = 2 if in_colab else 0

run_keys = [("0",str(i),f"Sherlock{j}","1") for j in range(1,8) for i in range(1, num_chapters[j-1])]
all_data = LibriBrainSpeech(
  data_path=f"{base_path}/data/",
  include_run_keys = run_keys,
  tmin=0.0,
  tmax=0.8,
  preload_files = True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


(…)-0_ses-5_task-Sherlock5_run-1_events.tsv: 0.00B [00:00, ?B/s]

(…)-0_ses-4_task-Sherlock5_run-1_events.tsv: 0.00B [00:00, ?B/s]

Sherlock1/derivatives/serialised/sub-0_s(…):   0%|          | 0.00/400M [00:00<?, ?B/s]

Sherlock3/derivatives/serialised/sub-0_s(…):   0%|          | 0.00/1.14G [00:00<?, ?B/s]

...

(…)-0_ses-3_task-Sherlock7_run-1_events.tsv: 0.00B [00:00, ?B/s]

Sherlock7/derivatives/serialised/sub-0_s(…):   0%|          | 0.00/371M [00:00<?, ?B/s]

Sherlock5/derivatives/serialised/sub-0_s(…):   0%|          | 0.00/523M [00:00<?, ?B/s]

(…)-0_ses-1_task-Sherlock6_run-1_events.tsv: 0.00B [00:00, ?B/s]

Done!
calculated stats for:  ('0', '1', 'Sherlock1', '1')
calculated stats for:  ('0', '2', 'Sherlock1', '1')
calculated stats for:  ('0', '3', 'Sherlock1', '1')
calculated stats for:  ('0', '4', 'Sherlock1', '1')
calculated stats for:  ('0', '5', 'Sherlock1', '1')
calculated stats for:  ('0', '6', 'Sherlock1', '1')
calculated stats for:  ('0', '7', 'Sherlock1', '1')
calculated stats for:  ('0', '8', 'Sherlock1', '1')
calculated stats for:  ('0', '1', 'Sherlock2', '1')
calculated stats for:  ('0', '2', 'Sherlock2', '1')
calculated stats for:  ('0', '3', 'Sherlock2', '1')
calculated stats for:  ('0', '4', 'Sherlock2', '1')
calculated stats for:  ('0', '5', 'Sherlock2', '1')
calculated stats for:  ('0', '6', 'Sherlock2', '1')
calculated stats for:  ('0', '7', 'Sherlock2', '1')
calculated stats for:  ('0', '8', 'Sherlock2', '1')
calculated stats for:  ('0', '9', 'Sherlock2', '1')
calculated stats for:  ('0', '10', 'Sherlock2', '1')
calculated stats for:  ('0', '11', 'Sherlock2', '1')
calc

## Load tsv event files and calculate class balances

In [ ]:
f = f"{base_path}/data/Sherlock1/derivatives/events/sub-0_ses-1_task-Sherlock1_run-1_events.tsv"
data = pd.read_csv(f, sep='\t')
data

In [ ]:
start_time = data["timemeg"].iloc[0]
end_time = data["timemeg"].iloc[-1]
tsv_data = data.copy()
tsv_data['timemeg'] = tsv_data['timemeg'].astype(float)
last_before = tsv_data[tsv_data['timemeg'] < start_time].iloc[-1:] # immediately preceding data
# window of interest
window_data = tsv_data[(tsv_data['timemeg'] >= start_time) & (tsv_data['timemeg'] <= end_time)]
filtered_data = pd.concat([last_before, window_data]).sort_values('timemeg')
if filtered_data.empty:
    filtered_data = pd.DataFrame({'timemeg': [start_time, end_time], 'speech_label': [0, 0]})
filtered_data['speech_label'] = 0
# combine word and phoneme labels for general 'speech' label
filtered_data.loc[filtered_data['kind'].isin(['word', 'phoneme']), 'speech_label'] = 1
filtered_data

In [5]:
tsv_file_paths = [f"{base_path}/data/Sherlock{j}/derivatives/events/sub-0_ses-{i}_task-Sherlock{j}_run-1_events.tsv" for j in range(1,8) for i in range(1, num_chapters[j-1])]

silence_n = {}
speech_n = {}
for f in tsv_file_paths:
  print(f)
  data = pd.read_csv(f, sep='\t')
  group_df = data.groupby(by="kind",as_index=False).count()
  curr_silence_n = group_df.iloc[1]["idx"]
  curr_speech_n = group_df.iloc[2]["idx"]
  silence_n[f] = curr_silence_n
  speech_n[f] = curr_speech_n

/content/data/Sherlock1/derivatives/events/sub-0_ses-1_task-Sherlock1_run-1_events.tsv
/content/data/Sherlock1/derivatives/events/sub-0_ses-2_task-Sherlock1_run-1_events.tsv
/content/data/Sherlock1/derivatives/events/sub-0_ses-3_task-Sherlock1_run-1_events.tsv
/content/data/Sherlock1/derivatives/events/sub-0_ses-4_task-Sherlock1_run-1_events.tsv
/content/data/Sherlock1/derivatives/events/sub-0_ses-5_task-Sherlock1_run-1_events.tsv
/content/data/Sherlock1/derivatives/events/sub-0_ses-6_task-Sherlock1_run-1_events.tsv
/content/data/Sherlock1/derivatives/events/sub-0_ses-7_task-Sherlock1_run-1_events.tsv
/content/data/Sherlock1/derivatives/events/sub-0_ses-8_task-Sherlock1_run-1_events.tsv
/content/data/Sherlock2/derivatives/events/sub-0_ses-1_task-Sherlock2_run-1_events.tsv
/content/data/Sherlock2/derivatives/events/sub-0_ses-2_task-Sherlock2_run-1_events.tsv
/content/data/Sherlock2/derivatives/events/sub-0_ses-3_task-Sherlock2_run-1_events.tsv
/content/data/Sherlock2/derivatives/events/

In [ ]:
speech_sum = 0
for k in speech_n:
  speech_sum += speech_n[k]
silence_sum = 0
for k in silence_n:
  silence_sum += silence_n[k]
print(speech_sum, silence_sum, (speech_sum+silence_sum), max(speech_sum,silence_sum)/(speech_sum+silence_sum))

In [6]:
import random
import torch
from torch.utils.data import DataLoader
import platform

# These are the sensors we identified as being particularly useful
SENSORS_SPEECH_MASK = [18, 20, 22, 23, 45, 120, 138, 140, 142, 143, 145,
                       146, 147, 149, 175, 176, 177, 179, 180, 198, 271, 272, 275]

class FilteredDataset(torch.utils.data.Dataset):
    """
    Parameters:
        dataset: LibriBrain dataset.
        limit_samples (int, optional): If provided, limits the length of the dataset to this
                          number of samples.
        speech_silence_only (bool, optional): If True, only includes segments that are either
                          purely speech or purely silence (with additional balancing).
        apply_sensors_speech_mask (bool, optional): If True, applies a fixed sensor mask to the sensor
                          data in each sample.
    """
    def __init__(self,
                 dataset,
                 limit_samples=None,
                 disable=False,
                 apply_sensors_speech_mask=True):
        self.dataset = dataset
        self.limit_samples = limit_samples
        self.apply_sensors_speech_mask = apply_sensors_speech_mask

        # These are the sensors we identified:
        self.sensors_speech_mask = SENSORS_SPEECH_MASK

        self.balanced_indices = list(range(len(dataset.samples)))
        # Shuffle the indices
        self.balanced_indices = random.sample(self.balanced_indices, len(self.balanced_indices))

    def __len__(self):
        """Returns the number of samples in the filtered dataset."""
        if self.limit_samples is not None:
            return self.limit_samples
        return len(self.balanced_indices)

    def __getitem__(self, index):
        # Map index to the original dataset using balanced indices
        original_idx = self.balanced_indices[index]
        if self.apply_sensors_speech_mask:
            sensors = self.dataset[original_idx][0][self.sensors_speech_mask]
        else:
            sensors = self.dataset[original_idx][0][:]
        label_from_the_middle_idx = self.dataset[original_idx][1].shape[0] // 2
        return [sensors, self.dataset[original_idx][1][label_from_the_middle_idx]]

print("Filtered dataset:")
all_data_filtered = FilteredDataset(all_data)
all_loader_filtered = DataLoader(all_data_filtered, batch_size=32, shuffle=True, num_workers=num_workers)
print(f"Filtered data contain {len(all_data_filtered)} samples")

Filtered dataset:
Filtered data contain 213257 samples


In [7]:
all_labels = []
for i_batch, (batch_data, batch_labels) in enumerate(all_loader_filtered):
  if len(batch_labels)<16:
    break
  all_labels.append(batch_labels[16])

In [14]:
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score

y_preds = np.ones(len(all_labels))

f1_macro = f1_score(all_labels, y_preds, average="macro")
f1_overall = f1_score(all_labels, y_preds)
print(f"Macro: {f1_macro}")
print(f"F1: {f1_overall}")

balanced_acc = balanced_accuracy_score(all_labels, y_preds)
roc_auc = roc_auc_score(all_labels, y_preds)
print(f"Balanced Acc: {balanced_acc}")
print(f"ROC AUC: {roc_auc}")

Macro: 0.43241631888254833
F1: 0.8648326377650967
Balanced Acc: 0.5
ROC AUC: 0.5


In [12]:
print(len(all_labels))

6664
